# แบบทดสอบ Fullstack Developer (Specialized HR/Payroll)

---

# ส่วนที่ 1 — Technical & Problem Solving

## ข้อ 1: The Buggy Payroll Logic

วิเคราะห์โค้ดต้นฉบับ และระบุจุดผิดพลาดร้ายแรง 3 จุด:

**บัก 1: Floating Point**
- ปัญหา: JavaScript ใช้ IEEE 754 float ทำให้ทศนิยมเพี้ยน
- ตัวอย่าง: `15000 * 0.05 = 749.999...` (ไม่ใช่ 750)
- แก้ไข: ใช้ `Decimal` library แทนทุกการคำนวณเงิน

**บัก 2: SQL Injection**
- ปัญหา: แปะ `${empId}` ตรงใน query string โดยไม่กรอง
- ตัวอย่าง: `empId = "1; DROP TABLE salaries--"` → ลบตารางได้ทันที
- แก้ไข: Parameterized Query (`?`) + validate type ก่อนใช้

**บัก 3: Race Condition**
- ปัญหา: รัน processPayroll() พร้อมกัน 2 ครั้ง → double payment
- ตัวอย่าง: Request A และ B อ่าน balance เดิมพร้อมกัน → บวกซ้ำ
- แก้ไข: DB Transaction + `SELECT FOR UPDATE` ล็อก row

## ข้อ 2: The Midnight Shift SQL

หาพนักงาน Night Shift วันที่ 19 มีนาคม 2026 ที่มาสาย (Clock-in หลัง 00:05)
รองรับกรณีสแกนก่อนเที่ยงคืน (23:55 วันที่ 18)

**Key Logic — ใช้ BETWEEN ครอบทั้งสองฝั่งเที่ยงคืน:**

```sql
WHERE s.shift_date = '2026-03-19'
  AND s.shift_type = 'NIGHT'
  AND a.clock_in BETWEEN '2026-03-18 23:45:00' AND '2026-03-19 08:00:00'
  AND a.clock_in > '2026-03-19 00:05:00'
```

**ผลลัพธ์:**

| พนักงาน | clock_in | สถานะ |
|---------|----------|-------|
| สมชาย | 2026-03-19 00:10 | สาย |
| สมหญิง | 2026-03-18 23:55 | ตรงเวลา ← สแกนก่อนเที่ยงคืน |
| มานะ | 2026-03-19 00:20 | สาย |
| ประสิทธิ์ | 2026-03-19 01:00 | สาย |
| รัตนา | 2026-03-19 00:03 | ตรงเวลา |

## ข้อ 3: System Design & Audit Trail

ออกแบบ REST API และ Database Schema สำหรับระบบแก้ไขเงินเดือนย้อนหลัง

**REST API:**

| Method | Endpoint | หน้าที่ |
|--------|----------|--------|
| POST | `/api/payroll/:id/amendments` | HR ขอแก้ไข |
| PATCH | `/api/payroll/amendments/:id/approve` | MANAGER/FINANCE อนุมัติ |
| PATCH | `/api/payroll/amendments/:id/reject` | MANAGER/FINANCE ปฏิเสธ |
| GET | `/api/payroll/:id/audit` | ดู audit log |

**Guards (การป้องกัน):**
- 1: เฉพาะ HR แก้เงินเดือนได้
- 2: ห้ามแก้เงินเดือนตัวเอง (self-edit)
- 3: บังคับระบุเหตุผลทุกครั้ง
- 4: ผู้อนุมัติต้องเป็น MANAGER หรือ FINANCE เท่านั้น

**Audit Trail (append-only):**
- บันทึก `old_value`, `new_value`, `changed_by`, `reason` ทุกครั้ง
- ห้าม UPDATE/DELETE ใน `payroll_audit`
- ผู้แก้ กับ ผู้อนุมัติ ต้องไม่ใช่คนเดียวกัน

---
# ส่วนที่ 2 — Business Analysis & Data Migration

## ข้อ 4: ManufacturePlus Scenario

บริษัทมีพนักงาน 800 คน คำนวณเงินเดือนด้วย Excel มา 10 ปี

### Functional Requirements: ลดการเดินมาถาม HR เรื่องวันลา

**FR-1: ดูวันลาคงเหลือ + ประวัติการลา**
- วันลาคงเหลือแยกประเภท (พักร้อน / ป่วย / กิจ) แบบ Real-time
- ประวัติการลาย้อนหลังทั้งหมด

**FR-2: ขอลาออนไลน์**
- เลือกประเภทลา / วันที่ / เหตุผล แล้ว Submit
- หัวหน้าได้รับ Notification → อนุมัติ/ปฏิเสธออนไลน์
- ระบบตัดวันลาคงเหลืออัตโนมัติเมื่ออนุมัติ

**FR-3: แจ้งเตือนผลการอนุมัติ (Notification)**
- พนักงานได้รับแจ้งเตือนทันทีเมื่อคำขอถูกอนุมัติหรือปฏิเสธ
- ดูสถานะคำขอได้ตลอดเวลา (Pending / Approved / Rejected)

### เทคนิค Data Cleansing: จัดการชื่อผิด และ Employee ID ซ้ำ

**ข้อมูลดิบตัวอย่าง (สกปรกจาก Excel 10 ปี):**

| emp_id | emp_name | ปัญหา |
|--------|----------|-------|
| EMP001 | สมชาย  ใจดี | whitespace ซ้ำในชื่อ |
| EMP002 | สมหญิง ใจงาม | — |
| EMP002 | สมหญิง ใจงาม | **ซ้ำ 100%** |
| emp004 | วิไล สวยงาม | **ID ตัวพิมพ์เล็ก** |
| EMP005 | ประสิทธิ์ ขยัน | — |
| EMP005 | ประสิทธิ เขยัน | **ซ้ำ + ชื่อสะกดต่างกัน** |

---

**1. Standardize Employee ID**
- Format: `EMP000`
- Uppercase + Trim → `emp004` → `EMP004`
- ID ผิด format → Flag ให้ HR แก้มือ

**2. ชื่อพนักงาน**
- ลบ whitespace ซ้ำ + Trim → `"สมชาย  ใจดี"` → `"สมชาย ใจดี"`
- ใช้ Fuzzy Matching จับคู่ชื่อที่สะกดต่างกันเล็กน้อย → Flag ให้ตรวจสอบ

**3. Duplicate Employee ID**

| กรณี | วิธีจัดการ |
|------|-----------|
| ซ้ำ + ข้อมูลเหมือน 100% | ลบ duplicate ออกได้เลย |
| ซ้ำ + ข้อมูลต่างกัน | Quarantine → HR ยืนยันก่อน Import |

**4. Rule**
- ห้าม Auto-fix โดยไม่มีมนุษย์ยืนยัน
- Export รายงาน Data Quality ให้ HR ตรวจก่อน Import จริง
- ทำ Dry-run ใน Staging environment ก่อนเสมอ

---
## ข้อ 5: Database Modeling for Shifts 

ออกแบบ Database Schema สำหรับ Shift Swapping พร้อม Approval Workflow และคำนวณเบี้ยเลี้ยงกะดึกถูกต้อง

In [ ]:
schema_shift = """
=== Database Schema: Shift Swapping System ===

-- ตารางกะงาน
CREATE TABLE shift_types (
    shift_type_id  SERIAL PRIMARY KEY,
    name           TEXT NOT NULL,           -- 'MORNING' | 'EVENING' | 'NIGHT'
    start_time     TIME NOT NULL,           -- 08:00 / 16:00 / 00:00
    end_time       TIME NOT NULL,           -- 16:00 / 00:00 / 08:00
    allowance      NUMERIC(8,2) DEFAULT 0,  -- เบี้ยเลี้ยง (Night = สูงกว่า)
    crosses_midnight BOOLEAN DEFAULT FALSE  -- Night Shift ข้ามวัน
);

-- ตารางกะที่ถูก assign ให้พนักงานแต่ละวัน
CREATE TABLE employee_shifts (
    shift_id       SERIAL PRIMARY KEY,
    emp_id         INTEGER NOT NULL,
    shift_date     DATE NOT NULL,
    shift_type_id  INTEGER NOT NULL REFERENCES shift_types(shift_type_id),
    is_original    BOOLEAN DEFAULT TRUE,    -- TRUE = กะเดิม, FALSE = กะที่ได้จากการสลับ
    UNIQUE (emp_id, shift_date)
);

-- ตารางคำขอสลับกะ
CREATE TABLE shift_swap_requests (
    swap_id         SERIAL PRIMARY KEY,
    requester_id    INTEGER NOT NULL,       -- คนที่ขอสลับ
    acceptor_id     INTEGER NOT NULL,       -- คนที่รับสลับ
    requester_shift INTEGER NOT NULL REFERENCES employee_shifts(shift_id),
    acceptor_shift  INTEGER NOT NULL REFERENCES employee_shifts(shift_id),
    status          TEXT DEFAULT 'PENDING', -- PENDING | APPROVED | REJECTED | CANCELLED
    requested_at    TIMESTAMPTZ DEFAULT NOW(),
    reviewed_by     INTEGER,                -- หัวหน้าที่อนุมัติ
    reviewed_at     TIMESTAMPTZ,
    reject_reason   TEXT
);

-- View: คำนวณเบี้ยเลี้ยงจาก actual shift หลังสลับกะ
CREATE VIEW v_shift_allowance AS
SELECT
    es.emp_id,
    es.shift_date,
    st.name         AS shift_name,
    st.allowance    AS night_allowance,
    es.is_original
FROM employee_shifts es
JOIN shift_types st ON st.shift_type_id = es.shift_type_id;
"""


### อธิบาย Schema

**3 ตารางหลัก:**

| ตาราง | หน้าที่ |
|-------|--------|
| `shift_types` | เก็บประเภทกะ (MORNING/EVENING/NIGHT) พร้อมเวลาและเบี้ยเลี้ยง |
| `employee_shifts` | กะที่ assign ให้พนักงานแต่ละวัน — `is_original` บอกว่ากะเดิมหรือกะที่ได้จากการสลับ |
| `shift_swap_requests` | คำขอสลับกะ พร้อม status (PENDING / APPROVED / REJECTED) |

**Key Design:**
- เบี้ยเลี้ยงคำนวณจาก **actual shift ที่ทำจริง** ไม่ใช่กะเดิม → ถ้าสลับกะดึกมาทำ ได้เบี้ยกะดึกเต็ม
- `UNIQUE (emp_id, shift_date)` → พนักงาน 1 คน มีได้แค่ 1 กะต่อวัน
- Approval workflow: หัวหน้า review ผ่าน `reviewed_by` + `reviewed_at`

---
# ส่วนที่ 3 — AI Integration & Supervision

## ข้อ 6: Smart Policy Bot

เขียน Prompt สำหรับ AI ตอบคำถามเรื่อง "สิทธิ์การลาสะสม"

### System Prompt


คุณคือ HR Policy Assistant ของบริษัท สำหรับตอบคำถามเรื่องสิทธิ์การลาเท่านั้น

กฎการลาของบริษัท (ห้ามเบี่ยงเบน):
- ลาพักร้อน: 10 วัน/ปี
- สะสมได้สูงสุด: 15 วัน (รวมยอดยกมาจากปีก่อน)
- ต้องแจ้งล่วงหน้าอย่างน้อย: 3 วันทำการ
- วันลาที่ยังไม่ได้ใช้เกิน 15 วัน: หมดอายุโดยอัตโนมัติทุกวันที่ 31 ธันวาคม

วิธีตอบ:
1. ตอบตามกฎข้างต้นเท่านั้น ห้ามคาดเดาหรือแต่งเติม
2. ถ้าคำถามอยู่นอกขอบเขตนโยบายลา ให้บอก: "กรุณาติดต่อ HR โดยตรง"
3. ถ้าต้องคำนวณวันลา ให้แสดงวิธีคิดทีละขั้นตอนอย่างชัดเจน
4. ห้ามให้คำแนะนำที่ขัดกับกฎบริษัทไม่ว่ากรณีใด
5. ตอบเป็นภาษาเดียวกับที่พนักงานถาม


### Prompt Guard: ตรวจสอบคำตอบ AI ก่อนส่งให้พนักงาน

ก่อนส่งคำตอบของ AI ให้พนักงาน ระบบจะตรวจสอบ 3 จุด:

**Guard 1: วันลาเกิน cap หรือไม่**
- ถ้า AI แนะนำวันลาเกิน 15 วัน → Reject

**Guard 2: ยอดวันลาถูกต้องหรือไม่**
- ตรวจว่าตัวเลขที่ AI บอกตรงกับยอดจริงในระบบ

**Guard 3: วันแจ้งล่วงหน้าถูกต้องหรือไม่**
- ถ้า AI บอก "แจ้งล่วงหน้า 1–2 วัน" → Reject (กฎบริษัทคือ 3 วัน)

**ผลลัพธ์:**

| กรณี | AI บอกว่า | ผล |
|------|-----------|-----|
| ถูกต้อง | "วันลาคงเหลือ 7 วัน แจ้งล่วงหน้า 3 วัน" | ✅ ผ่าน |
| ผิดพลาด | "ลาได้ 20 วัน แจ้งล่วงหน้า 1 วันก็พอ" | ❌ → "กรุณาติดต่อ HR โดยตรง" |

---
## ข้อ 7: AI Feature Design — Payroll Anomaly Detector

### Architecture: Payroll Anomaly Detector

```
┌──────────────────────────────────────────────────────────┐
│                     DATA LAYER                           │
│  payroll_db ──► ETL Pipeline ──► Feature Store           │
│  (PostgreSQL)   (dbt/Airflow)    (anonymized + masked)   │
└──────────────────────────────────────────────────────────┘
                         │
                         ▼
┌──────────────────────────────────────────────────────────┐
│                   DETECTION LAYER                        │
│                                                          │
│  Rule-Based Engine          ML Model                     │
│  (เร็ว, อธิบายได้)           (Isolation Forest)          │
│  - OT > 80 ชม./เดือน        - Pattern ผิดปกติที่ Rule    │
│  - Salary เพิ่ม > 20%        จับไม่ได้                   │
│  - Net pay ติดลบ                                         │
│  - แก้เงินเดือนตัวเอง                                     │
│           │                       │                     │
│           └──────────┬────────────┘                     │
│                      ▼                                   │
│              Risk Score (0–100)                          │
└──────────────────────────────────────────────────────────┘
                         │
                         ▼
┌──────────────────────────────────────────────────────────┐
│                   ACTION LAYER                           │
│  Score < 40  → Log only                                  │
│  Score 40–70 → Alert HR Manager                          │
│  Score > 70  → Freeze payroll + Alert CFO                │
└──────────────────────────────────────────────────────────┘
```

### Anomaly Rules + Risk Scoring

**Rule-Based Detection:**

| Rule | เงื่อนไข | Risk Score |
|------|---------|-----------|
| OT สูงผิดปกติ | OT > 80 ชม./เดือน | +30 |
| เงินเดือนพุ่งขึ้น | เพิ่มมากกว่า 20% จากเดือนก่อน | +50 |
| Net pay ติดลบ | net_pay < 0 (Error ในการคำนวณ) | +80 |
| แก้เงินเดือนตัวเอง | self_edit = true | +90 |
| จ่ายซ้ำ | พนักงานคนเดียวได้รับเงิน 2 ครั้ง/เดือน | +70 |

**Action ตาม Risk Score:**

| Score | Action |
|-------|--------|
| < 40 | Log only — บันทึกไว้ตรวจสอบภายหลัง |
| 40–70 | Alert HR Manager — แจ้งเตือนให้ตรวจสอบ |
| > 70 | **Freeze payroll** + Alert CFO + บังคับ Audit ก่อนจ่าย |

**ตัวอย่างผลลัพธ์:**

| พนักงาน | Risk Score | Anomaly | Action |
|---------|-----------|---------|--------|
| สมชาย | 0 | ปกติ | LOG |
| สมหญิง | 30 | OT 95 ชม. | LOG |
| IT Admin | 100 | เงินเดือน +25% + แก้ตัวเอง | **FREEZE** |
| มานะ | 80 | Net pay ติดลบ | **FREEZE** |

### Data Privacy: จัดการข้อมูลก่อนส่งให้ AI Model

ก่อนส่งข้อมูลพนักงานให้ AI วิเคราะห์ ต้อง Anonymize ก่อนเสมอ:

**สิ่งที่ลบออก (PII = ข้อมูลที่ระบุตัวตนได้):**
- ชื่อ-นามสกุล, เลขบัตรประชาชน, เบอร์โทร, ที่อยู่
- เงินเดือนจริง (แปลงเป็น Salary Band แทน)

**สิ่งที่ส่งให้ AI (Behavioral Pattern เท่านั้น):**

| Field | วิธีจัดการ |
|-------|-----------|
| emp_id | Hash (SHA-256 + salt) → ไม่สามารถย้อนรอยได้ |
| salary | แปลงเป็น Band A/B/C ตามช่วงเงินเดือน |
| department | ส่งเป็น code แทนชื่อแผนก |
| ot_hours | ส่งตรง (ไม่ใช่ PII) |
| salary_change_pct | ส่งตรง (ไม่ใช่ PII) |

**Hash + Mapping Table:**
- AI ได้รับแค่ hash เช่น `x9q2mn...` แทน `EMP001` → ไม่รู้ว่าเป็นใคร
- Mapping table (`hash ↔ emp_id`) เก็บใน Secure DB ที่ AI เข้าไม่ถึง
- เมื่อ AI พบ anomaly → HR เอา hash ไป lookup → รู้ว่าต้องดำเนินการกับพนักงานคนไหน

**Measures เพิ่มเติม:**
- ใช้ AI Model ที่รันบน Private Cloud (ไม่ส่งข้อมูลออกนอก)
- บันทึก Audit Log ทุกครั้งที่ข้อมูลถูกส่งให้ AI
- Data Minimization — ส่งเฉพาะ field ที่ AI ต้องการจริงๆ